In [2]:
# electricity
from training_utilities_2nd_part import *
from variables_to_specify_electricity import *
df, columns_to_normalize, elect_target_col, forecast_avg_target_col_name, avg_target_col_name, No_of_datapoints_in_one_day, start_date, end_date, delta, one_month_days, out_columns, elect_drop_columnss, elect_windows, index_of_one_month, one_month_window_size, date_col_name = variables_to_specify_electricity()

df = df.dropna().reset_index(drop=True)
df = df.drop(range(1344)).reset_index(drop=True)
electricity_df = df
elect_time_steps = 1
electricity_df

,date,day,period,nswprice,nswdemand,vicprice,vicdemand,transfer,class
0,1970-01-01,6,0.000000,0.085565,0.541803,0.003467,0.422915,0.414912,1
1,1970-01-01,6,0.021277,0.085565,0.506992,0.003467,0.422915,0.414912,1
2,1970-01-01,6,0.042553,0.078209,0.477834,0.003467,0.422915,0.414912,1
3,1970-01-01,6,0.063830,0.064519,0.415204,0.003467,0.422915,0.414912,0
4,1970-01-01,6,0.085106,0.064519,0.351532,0.003467,0.422915,0.414912,0
...,...,...,...,...,...,...,...,...,...
42763,1970-01-01,7,0.914894,0.044224,0.340672,0.003033,0.255049,0.405263,0
42764,1970-01-01,7,0.936170,0.044884,0.355549,0.003072,0.241326,0.420614,0
42765,1970-01-01,7,0.957447,0.043593,0.340970,0.002983,0.247799,0.362281,0
42766,1970-01-01,7,0.978723,0.066651,0.329366,0.004630,0.345417,0.206579,1


# stationary

In [4]:
elect_len_of_training_data_of_stationary_model = 14*No_of_datapoints_in_one_day

stationary_model = new_copied_lstm_statinary(electricity_df, elect_len_of_training_data_of_stationary_model, elect_target_col, elect_drop_columnss, elect_time_steps)

Epoch 1/10


2025-04-10 18:01:28.801514: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


21/21 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 8.5435e-04
Epoch 2/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 6.6288e-04
Epoch 3/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 6.5670e-04
Epoch 4/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 6.5633e-04
Epoch 5/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 6.5331e-04
Epoch 6/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 6.5546e-04
Epoch 7/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 6.5579e-04
Epoch 8/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 6.5488e-04
Epoch 9/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 6.5387e-04
Epoch 10/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 6.5419e-04
X_train shape: (672, 1, 4)
y_train shape: (672,)
X_train mean: 0.33639112
X_train std: 0.21287267
y_train mean: 0.07440636
y_train std: 0.028060013
y_train min: 0.041191
y_train max: 0.179236
total_test_error is : 0.002391618
total_test_error_mae is:  0.034544583
total_time is:  8.208923167
train time is :  0
Model 

 # Model reuse

In [3]:
daily_df_avg = get_elect_daily_avg(electricity_df, No_of_datapoints_in_one_day, elect_target_col, avg_target_col_name)

seasonality_periods_acf_ls, seasonality_periods_acf, segmented_daily_df_avg, filtered_most_similar_dict_wass, filtered_most_similar_dict_tvd, forecast_daily_df_avg, segmented_forecast_daily_df_avg, filtered_forecasted_most_similar_dict_wass, filtered_forecasted_most_similar_dict_tvd = get_seasonality_segments_and_similarities(daily_df_avg, avg_target_col_name, forecast_avg_target_col_name, 14)

Detected seasonality periods (ACF): [  4   7  10  14  17  21  23  27  34  42  49  53  56  58  63  68  72  76
  80  85  92  97 104 107 112 114 121]
median_value is:  58


In [7]:
ratio_wass = len(filtered_most_similar_dict_wass)/segmented_daily_df_avg.shape[1]
ratio_tvd = len(filtered_most_similar_dict_tvd)/segmented_daily_df_avg.shape[1]
ratio_forecasted_wass = len(filtered_forecasted_most_similar_dict_wass)/segmented_daily_df_avg.shape[1]
ratio_forecasted_tvd = len(filtered_forecasted_most_similar_dict_tvd)/segmented_daily_df_avg.shape[1]

print('ratio_wass:', ratio_wass)
print('ratio_tvd:', ratio_tvd)
print('ratio_forecasted_wass:', ratio_forecasted_wass)
print('ratio_forecasted_tvd:', ratio_forecasted_tvd)

ratio_wass: 0.746031746031746
ratio_tvd: 0.8253968253968254
ratio_forecasted_wass: 0.8253968253968254
ratio_forecasted_tvd: 0.6349206349206349


In [8]:
df_copy = electricity_df[[elect_target_col]]
target_col = elect_target_col
time_steps = elect_time_steps

df_copy['date'] = pd.to_datetime(df_copy.index)
multiplier = No_of_datapoints_in_one_day
x = 14* multiplier
window_len_=[x]


drift_results_df_ls = []

for i in window_len_:
    start_drift_detection_time = timeit.default_timer()
    drift_results_df = detect_drift_univariate(
        df_copy,
        target_col=elect_target_col,
        window_lengths=window_len_,
        arima_order=(1, 0, 0)
    )
    drift_results_df_ls.append(drift_results_df)
    drift_detection_time = timeit.default_timer() - start_drift_detection_time
    num_true = drift_results_df['drift_detected'].sum()
    print("i is: ", i, " and the Number of True values in 'drift_detected':", num_true, " total number of rows are : ", len(drift_results_df))
    print("drift detection time is: ", drift_detection_time)
    drift_results_df = drift_results_df_ls[0]
    drift_indices = list(drift_results_df.index[drift_results_df['drift_detected']])
    print("indices are: ", drift_indices)

Fold 0: Train size=134, Test size=134
Fold 1: Train size=268, Test size=134
Fold 2: Train size=402, Test size=134
Fold 3: Train size=536, Test size=134
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=134, Test size=134
Fold 1: Train size=268, Test size=134
Fold 2: Train size=402, Test size=134
Fold 3: Train size=536, Test size=134
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=134, Test size=134
Fold 1: Train size=268, Test size=134
Fold 2: Train size=402, Test size=134
Fold 3: Train size=536, Test size=134
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=134, Test size=134
Fold 1: Train size=268, Test size=134
Fold 2: Train size=402, Test size=134
Fold 3: Train size=536, Test size=134
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=134, Test size=134
Fold 1: Train size=268, Test size=134
Fold 2: Train size=402, Test size=134
Fold 3: Train size=536, Test size=134
Skipping fold 4: Insufficien

In [8]:
eval_df_monthly2, eval_df_monthly, avg_ml_storage1= new_copied_lstm_reuse_with_hptuning_no_while_loop_with_drift(filtered_most_similar_dict_wass, stationary_model, elect_len_of_training_data_of_stationary_model, electricity_df, "SA", elect_target_col, elect_drop_columnss, elect_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices)

print("total reuse reduced count is: ",total_reduced_count_of_retrainings(filtered_most_similar_dict_wass))

window is:  672
Epoch 1/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 8.3851e-04
Epoch 2/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 7.0810e-04
Epoch 3/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 6.8994e-04
Epoch 4/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 6.9291e-04
Epoch 5/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 6.9390e-04
Epoch 6/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 6.9303e-04
Epoch 7/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 6.9367e-04
Epoch 8/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 6.9066e-04
Epoch 9/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 6.9283e-04
Epoch 10/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 6.9438e-04
i/window is :  1.0
Epoch 1/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0041
Epoch 2/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 8.4628e-04
Epoch 3/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 7.0166e-04
Epoch 4/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss:

In [9]:
eval_df_monthly2, eval_df_monthly, avg_ml_storage2= new_copied_lstm_reuse_with_hptuning_no_while_loop_with_drift(filtered_most_similar_dict_tvd, stationary_model, elect_len_of_training_data_of_stationary_model, electricity_df, "SA", elect_target_col, elect_drop_columnss, elect_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices)

print("total reuse reduced count is: ",total_reduced_count_of_retrainings(filtered_most_similar_dict_tvd))

window is:  672
Epoch 1/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - loss: 8.3851e-04
Epoch 2/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 7.0810e-04
Epoch 3/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 6.8994e-04
Epoch 4/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 6.9291e-04
Epoch 5/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 6.9390e-04
Epoch 6/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 6.9303e-04
Epoch 7/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 6.9367e-04
Epoch 8/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 6.9066e-04
Epoch 9/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 6.9283e-04
Epoch 10/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 6.9438e-04
i/window is :  1.0
Epoch 1/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.0041
Epoch 2/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 8.4628e-04
Epoch 3/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 7.0166e-04
Epoch 4/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss:

In [10]:
eval_df_monthly2, eval_df_monthly, avg_ml_storage3= new_copied_lstm_reuse_with_hptuning_no_while_loop_with_drift(filtered_forecasted_most_similar_dict_wass, stationary_model, elect_len_of_training_data_of_stationary_model, electricity_df, "ES", elect_target_col, elect_drop_columnss, elect_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices)

print("total reuse reduced count is: ",total_reduced_count_of_retrainings(filtered_forecasted_most_similar_dict_wass))

window is:  672
Epoch 1/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - loss: 8.3851e-04
Epoch 2/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 7.0810e-04
Epoch 3/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 6.8994e-04
Epoch 4/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 6.9291e-04
Epoch 5/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 6.9390e-04
Epoch 6/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 6.9303e-04
Epoch 7/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 6.9367e-04
Epoch 8/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 6.9066e-04
Epoch 9/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 6.9283e-04
Epoch 10/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 6.9438e-04
Epoch 1/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - loss: 0.0041
Epoch 2/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 8.4628e-04
Epoch 3/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 7.0166e-04
Epoch 4/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 6.8784e-04
Epoch 5

In [11]:
eval_df_monthly2, eval_df_monthly, avg_ml_storage4= new_copied_lstm_reuse_with_hptuning_no_while_loop_with_drift(filtered_forecasted_most_similar_dict_tvd, stationary_model, elect_len_of_training_data_of_stationary_model, electricity_df, "ES", elect_target_col, elect_drop_columnss, elect_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices)

print("total reuse reduced count is: ",total_reduced_count_of_retrainings(filtered_forecasted_most_similar_dict_tvd))

window is:  672
Epoch 1/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - loss: 8.3851e-04
Epoch 2/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 7.0810e-04
Epoch 3/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 6.8994e-04
Epoch 4/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 6.9291e-04
Epoch 5/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 6.9390e-04
Epoch 6/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 6.9303e-04
Epoch 7/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 6.9367e-04
Epoch 8/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 6.9066e-04
Epoch 9/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 6.9283e-04
Epoch 10/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 6.9438e-04
Epoch 1/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - loss: 0.0041
Epoch 2/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 8.4628e-04
Epoch 3/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 7.0166e-04
Epoch 4/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 6.8784e-04
Epoch 5

In [12]:
avg_ml_storage_reuse = (avg_ml_storage1+avg_ml_storage2+avg_ml_storage3+avg_ml_storage4)/4
print("avg_ml_storage_reuse is : ", avg_ml_storage_reuse)

avg_ml_storage_reuse is :  0.059329986572265625


# informed retraining

In [13]:
lstm_informed_update(stationary_model,electricity_df, elect_target_col, elect_drop_columnss,time_steps, seasonality_periods_acf,No_of_datapoints_in_one_day, drift_indices)

window is:  672
Epoch 1/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - loss: 8.3851e-04
Epoch 2/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 7.0810e-04
Epoch 3/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 6.8994e-04
Epoch 4/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 6.9291e-04
Epoch 5/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 6.9390e-04
Epoch 6/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 6.9303e-04
Epoch 7/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 6.9367e-04
Epoch 8/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 6.9066e-04
Epoch 9/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 6.9283e-04
Epoch 10/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 6.9438e-04
Model Type: Sequential
Storage Required: 0.06 MB
window is:  1344
window is:  2016
Epoch 1/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - loss: 0.0035
Epoch 2/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 7.1256e-04
Epoch 3/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 5.663

# periodical

In [9]:
mean_mse_per_window, min_index, mean_mae_per_window = periodical_lstm_training(electricity_df, elect_target_col, elect_time_steps, elect_windows, elect_drop_columnss)

window size is :  240
Epoch 1/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 43ms/step - loss: 4.2648e-04
Epoch 2/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 3.9529e-04
Epoch 3/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 3.6905e-04
Epoch 4/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 3.1764e-04
Epoch 5/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 3.3368e-04
Epoch 6/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 3.1965e-04
Epoch 7/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 3.2665e-04
Epoch 8/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 3.2204e-04
Epoch 9/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 3.2292e-04
Epoch 10/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 3.2310e-04
Model Type: Sequential
Storage Required: 0.06 MB
Epoch 1/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.0053
Epoch 2/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0020
Epoch 3/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 9.2511e-04
Epoch 4/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - l